<a href="https://colab.research.google.com/github/seo55555/dkdltmxl/blob/main/code/05_table1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 아이스티 (1조)

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 170)
np.set_printoptions(suppress=True)

In [5]:
import os
DATA, BOOK = 'knhanes_like_week4.csv', 'codebook_week4.csv'

if not os.path.exists(DATA):
    from google.colab import files
    files.upload()

df = pd.read_csv(DATA)
w  = df['wt'].to_numpy()
ST, PS = df['kstrata'].to_numpy(), df['psu'].to_numpy()

print(f'n = {len(df):,}  |  조사구 {df.psu.nunique()}개  |  층 {df.kstrata.nunique()}개')
print(f'가중치 합 = {w.sum():,.0f}')

Saving codebook_week4.csv to codebook_week4.csv
Saving knhanes_like_week4.csv to knhanes_like_week4.csv
n = 5,367  |  조사구 263개  |  층 18개
가중치 합 = 450,899


---
# 실습 — Table 1 완성

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# -------------------------------------------------------------------------
# 1. svymean_domain 함수 정의 (결측치 처리 로직 반영)
# -------------------------------------------------------------------------
def svymean_domain(y, w, ST, PS, dom):
    """
    도메인(dom) 내에서 복합표본 가중평균과 표준오차(SE)를 계산하는 함수
    """
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    ST = np.asarray(ST)
    PS = np.asarray(PS)

    # 교수님이 제안해주신 의견
    dom = np.asarray(dom, dtype=bool) & ~np.isnan(y)    # 결측은 도메인에서 제외
    y = np.nan_to_num(y)    # NaN → 0 (가중치가 0이라 결과에 영향 없음)

    # 도메인 가중치 적용
    w_dom = w * dom
    sum_w = np.sum(w_dom)

    if sum_w == 0:
        return np.nan, np.nan

    # 가중평균 (Estimate)
    mean_est = np.sum(w_dom * y) / sum_w

    # 층화/집단 표본추출에 따른 분산(SE) 계산 (Taylor Series Linearization)
    residuals = dom * (y - mean_est)

    psu_vars = []
    unique_strata = np.unique(ST)

    for st in unique_strata:
        st_mask = (ST == st)
        unique_psus = np.unique(PS[st_mask])
        n_h = len(unique_psus)

        if n_h <= 1:
            continue

        psu_totals = []
        for psu in unique_psus:
            psu_mask = st_mask & (PS == psu)
            psu_totals.append(np.sum(w_dom[psu_mask] * residuals[psu_mask]))

        psu_totals = np.array(psu_totals)
        mean_psu = np.mean(psu_totals)
        var_h = (n_h / (n_h - 1)) * np.sum((psu_totals - mean_psu) ** 2)
        psu_vars.append(var_h)

    var_total = np.sum(psu_vars) / (sum_w ** 2)
    se_est = np.sqrt(var_total)

    return mean_est, se_est

# -------------------------------------------------------------------------
# 2. 노출군 / 비노출군 정의 (도메인 정의)
# -------------------------------------------------------------------------
# smk: 1=현재흡연, 2=과거흡연, 3=비흡연
smk_exp = (df.smk == 1)                        # 노출군 (현재흡연)
smk_unexp = df.smk.isin([2, 3])                # 비노출군 (비현재흡연: 과거흡연 + 비흡연)

DOMS = {
    '전체': np.ones(len(df), dtype=bool),
    '노출군(현재흡연)': smk_exp,
    '비노출군(비현재흡연)': smk_unexp
}

# 각 집단별 관측 수 (n) 계산
n_total = len(df)
n_exp = int(smk_exp.sum())
n_unexp = int(smk_unexp.sum())

# -------------------------------------------------------------------------
# 3. Table 1 행 생성 함수
# -------------------------------------------------------------------------
def row_cont(label, var):
    """연속형 변수: 전체 n, 그룹별 n 및 가중 평균 (SE)"""
    valid_mask = ~df[var].isna()
    n_obs = int(df.loc[valid_mask, var].count())

    # 그룹별 유효 관측 수 계산
    n_obs_exp = int(df.loc[valid_mask & smk_exp, var].count())
    n_obs_unexp = int(df.loc[valid_mask & smk_unexp, var].count())

    r = {'특성': label, 'n': f"{n_obs:,}", '노출군 n': f"{n_obs_exp:,}", '비노출군 n': f"{n_obs_unexp:,}"}

    for name, dom in DOMS.items():
        th, se = svymean_domain(df[var], w, ST, PS, dom)
        r[name] = f"{th:.1f} ({se:.2f})" if not np.isnan(th) else "nan (nan)"
    return r

def row_cat(label, mask, na_exclude=None):
    """범주형 변수: 전체 n, 그룹별 n 및 가중 %"""
    mask_arr = np.asarray(mask, dtype=bool)

    if na_exclude is not None:
        valid_mask = ~na_exclude
        mask_arr = mask_arr & valid_mask
    else:
        valid_mask = np.ones(len(df), dtype=bool)

    n_obs = int(np.sum(mask_arr & valid_mask))
    n_obs_exp = int(np.sum(mask_arr & valid_mask & smk_exp))
    n_obs_unexp = int(np.sum(mask_arr & valid_mask & smk_unexp))

    r = {'특성': label, 'n': f"{n_obs:,}", '노출군 n': f"{n_obs_exp:,}", '비노출군 n': f"{n_obs_unexp:,}"}

    for name, dom in DOMS.items():
        # 범주형 계산 시 유효 대상자에 대한 마스크 결합
        combined_dom = dom & valid_mask
        th, se = svymean_domain(mask_arr.astype(float), w, ST, PS, combined_dom)
        r[name] = f"{th * 100:.1f}%" if not np.isnan(th) else "nan%"
    return r

# -------------------------------------------------------------------------
# 4. Table 1 데이터프레임 생성 및 출력
# -------------------------------------------------------------------------
# 교육(edu)과 소득(income)의 무응답 코드 9 처리용 마스크
edu_valid = (df['edu'] != 9) & (~df['edu'].isna())
income_valid = (df['income'] != 9) & (~df['income'].isna())

t1_data = [
    row_cont('연령 (세)', 'age'),
    row_cat('  65세 이상', df.age >= 65),
    row_cat('남성', df.sex == 1),
    row_cat('읍면 거주', df.urban == 2),
    row_cont('BMI (kg/m²)', 'bmi'),
    row_cat('  비만 (BMI ≥ 25)', df.bmi >= 25),
    row_cont('수축기혈압 (mmHg)', 'sbp'),
    row_cat('고혈압', df.htn == 1),
    row_cat('현재흡연', df.smk == 1),
    row_cat('규칙적 운동', df.exercise == 2),
    row_cat('음주 여부 (음주군)', df.drink == 1), # drink 예시 변수
    row_cat('교육수준 (고졸 이상)', (df.edu >= 3), na_exclude=~edu_valid),
    row_cat('소득수준 (3분위 이상)', (df.income >= 3), na_exclude=~income_valid),
    row_cont('총콜레스테롤 (mg/dL)', 'chol'),
    row_cont('공복혈당 (mg/dL)', 'glucose')
]

t1 = pd.DataFrame(t1_data)

# 열 순서 배치
cols = ['특성', 'n', '전체', '노출군(현재흡연)', '비노출군(비현재흡연)']
t1_display = t1[['특성', 'n', '전체', '노출군(현재흡연)', '비노출군(비현재흡연)']].copy()
t1_display.columns = ['특성', 'n', '전체', f'노출군(현재흡연) [n={n_exp:,}]', f'비노출군(비현재흡연) [n={n_unexp:,}]']

# 표 출력 및 각주 표시
print(f"=== Table 1. 대상자의 일반적 특성 (전체 n={n_total:,}, 노출군 n={n_exp:,}, 비노출군 n={n_unexp:,}) ===")
display(t1_display)
print("\n* 각주: n은 결측치를 제외한 유효 관측 수, %는 가중 추정치, 연속형 변수는 가중평균(표준오차)으로 제시함.")
print("* 교육(edu) 및 소득(income) 변수의 무응답 코드(9)는 분석 대상에서 제외함.")
print("* 표본 설계정보: 가중치(wt), 층(kstrata), 집락(psu) 반영 도메인 분석 실시.")

=== Table 1. 대상자의 일반적 특성 (전체 n=5,367, 노출군 n=1,295, 비노출군 n=4,072) ===


,특성,n,전체,"노출군(현재흡연) [n=1,295]","비노출군(비현재흡연) [n=4,072]"
0,연령 (세),"5,367",49.1 (0.16),48.0 (0.39),49.5 (0.20)
1,65세 이상,"1,607",16.8%,14.4%,17.6%
2,남성,"2,719",49.0%,71.5%,41.9%
3,읍면 거주,"2,156",22.9%,23.6%,22.7%
4,BMI (kg/m²),"5,367",23.8 (0.05),23.3 (0.11),24.0 (0.06)
5,비만 (BMI ≥ 25),"2,035",36.5%,30.7%,38.4%
6,수축기혈압 (mmHg),"5,367",122.8 (0.45),124.0 (0.67),122.4 (0.46)
7,고혈압,"1,652",24.4%,27.2%,23.5%
8,현재흡연,"1,295",24.1%,100.0%,0.0%
9,규칙적 운동,"2,238",41.5%,42.1%,41.3%



* 각주: n은 결측치를 제외한 유효 관측 수, %는 가중 추정치, 연속형 변수는 가중평균(표준오차)으로 제시함.
* 교육(edu) 및 소득(income) 변수의 무응답 코드(9)는 분석 대상에서 제외함.
* 표본 설계정보: 가중치(wt), 층(kstrata), 집락(psu) 반영 도메인 분석 실시.
